In [1]:
import chromadb
client = chromadb.PersistentClient(path="./database.sqlite")


In [5]:
import chromadb.utils.embedding_functions as embedding_functions

# use directlye 
google_ef  = embedding_functions.GoogleGenerativeAiEmbeddingFunction(api_key="")

# pass documents to query for .add and .query
collection = client.get_or_create_collection(name="medicine", embedding_function=google_ef)


In [81]:
from pprint import pprint
query = "tell me the harm of the Advil cant be used  for"
results = collection.query(
    query_texts=[query], # Chroma will embed this for you
    n_results=10 # how many results to return
)
pprint(results)

{'data': None,
 'distances': [[0.43902236223220825,
                0.4432472586631775,
                0.4475964307785034,
                0.45077455043792725,
                0.4540291428565979,
                0.4554555118083954,
                0.46250995993614197,
                0.4683084487915039,
                0.47122204303741455,
                0.47170203924179077]],
 'documents': [['the drug name is Advil PM and the medical condition to take '
                'the drug is Insomnia and its side effect is chest pain '
                'spreading to your jaw or shoulder, sudden numbness or '
                'weakness on one side of the body, slurred speech, leg '
                'swelling, feeling short of breath. This medicine may cause '
                'serious side effects. Stop using this medicine and call your '
                'doctor at once if you have: any skin rash, no matter how '
                'mild; a light-headed feeling, like you might pass out; signs '
     

In [43]:
import google.generativeai as genai

In [67]:
# Replace with your Gemini API key
api_key = ""

# Configure Gemini
genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-pro")


In [83]:
retrieved_docs = results['documents'][0]  # Retrieved documents
safe_docs = [doc for doc in retrieved_docs if "misuse" not in doc and "illegal" not in doc]
print("Retrieved Documents:", safe_docs)

Retrieved Documents: ['the drug name is Advil PM and the medical condition to take the drug is Insomnia and its side effect is chest pain spreading to your jaw or shoulder, sudden numbness or weakness on one side of the body, slurred speech, leg swelling, feeling short of breath. This medicine may cause serious side effects. Stop using this medicine and call your doctor at once if you have: any skin rash, no matter how mild; a light-headed feeling, like you might pass out; signs of stomach bleeding--bloody or tarry stools, coughing up blood or vomit that looks like coffee grounds; kidney problems--little or no urinating, painful or difficult urination, swelling in your feet or ankles, feeling tired or short of breath; or liver problems-- nausea , upper stomach pain, itching, tired feeling, flu-like symptoms, loss of appetite, dark urine, clay-colored stools, jaundice (yellowing of the skin or eyes). Common side effects of Advil PM may include: drowsiness; day-time drowsiness, dizziness

In [85]:
# Combine the query and retrieved context
context = "\n".join(safe_docs)
full_prompt = f"""
Disclaimer: The following is for educational purposes only. Consult a licensed healthcare professional for medical advice.

Context:
{context}

Query:
{query}
"""

# Generate answer with Gemini/

response = model.generate_content(
    full_prompt,
    safety_settings=[
        {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "HIGH"}
    ],
)

print("Generated Answer:", response.text)

Generated Answer: I cannot perform a task based on a false premise. Advil can be used for many conditions, including pain, fever, and inflammation.


In [89]:
import pandas as pd

In [107]:
df = pd.read_csv("./drugLibTrain_raw.csv")

df.shape

clean_df = df.dropna()

cleaned_df = clean_df.iloc[:,:8]
cleaned_df.head()



,Unnamed: 0,urlDrugName,rating,effectiveness,sideEffects,condition,benefitsReview,sideEffectsReview
0,2202,enalapril,4,Highly Effective,Mild Side Effects,management of congestive heart failure,slowed the progression of left ventricular dys...,"cough, hypotension , proteinuria, impotence , ..."
1,3117,ortho-tri-cyclen,1,Highly Effective,Severe Side Effects,birth prevention,Although this type of birth control has more c...,"Heavy Cycle, Cramps, Hot Flashes, Fatigue, Lon..."
2,1146,ponstel,10,Highly Effective,No Side Effects,menstrual cramps,I was used to having cramps so badly that they...,Heavier bleeding and clotting than normal.
3,3947,prilosec,3,Marginally Effective,Mild Side Effects,acid reflux,The acid reflux went away for a few months aft...,"Constipation, dry mouth and some mild dizzines..."
4,1951,lyrica,2,Marginally Effective,Severe Side Effects,fibromyalgia,I think that the Lyrica was starting to help w...,I felt extremely drugged and dopey. Could not...


In [109]:
documents = []

for i, row in cleaned_df.iterrows():
    documents.append("the drug name is "+row[1]+" and the condition to take this drug is "+row[5]+" and its side effects is "+row[7])

for i in range(5):
    print(documents[i])

C:\Users\semoo\AppData\Local\Temp\ipykernel_21616\1085207674.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  documents.append("the drug name is "+row[1]+" and the condition to take this drug is "+row[5]+" and its side effects is "+row[7])


the drug name is enalapril and the condition to take this drug is management of congestive heart failure and its side effects is cough, hypotension , proteinuria, impotence , renal failure , angina pectoris , tachycardia , eosinophilic pneumonitis, tastes disturbances , anusease anorecia , weakness fatigue insominca weakness
the drug name is ortho-tri-cyclen and the condition to take this drug is birth prevention and its side effects is Heavy Cycle, Cramps, Hot Flashes, Fatigue, Long Lasting Cycles. It's only been 5 1/2 months, but i'm concidering changing to a different bc. This is my first time using any kind of bc, unfortunately due to the constant hassel, i'm not happy with the results.
the drug name is ponstel and the condition to take this drug is menstrual cramps and its side effects is Heavier bleeding and clotting than normal.
the drug name is prilosec and the condition to take this drug is acid reflux and its side effects is Constipation, dry mouth and some mild dizziness tha

In [115]:
ids=1692
id=0
for index in range(len(documents)):
    collection.add(
        documents=documents[id],
        ids=[str(ids)]
    )
    id = id + 1
    ids = ids+1
print("Data successfully added to ChromaDB!")

Insert of existing embedding ID: 1692
Add of existing embedding ID: 1692
Insert of existing embedding ID: 1693
Add of existing embedding ID: 1693
Insert of existing embedding ID: 1694
Add of existing embedding ID: 1694
Insert of existing embedding ID: 1695
Add of existing embedding ID: 1695
Insert of existing embedding ID: 1696
Add of existing embedding ID: 1696
Insert of existing embedding ID: 1697
Add of existing embedding ID: 1697
Insert of existing embedding ID: 1698
Add of existing embedding ID: 1698
Insert of existing embedding ID: 1699
Add of existing embedding ID: 1699
Insert of existing embedding ID: 1700
Add of existing embedding ID: 1700
Insert of existing embedding ID: 1701
Add of existing embedding ID: 1701
Insert of existing embedding ID: 1702
Add of existing embedding ID: 1702
Insert of existing embedding ID: 1703
Add of existing embedding ID: 1703
Insert of existing embedding ID: 1704
Add of existing embedding ID: 1704
Insert of existing embedding ID: 1705
Add of existi

KeyboardInterrupt: 

In [119]:
print(id)

363


In [125]:
index = ids+1
for index in range(len(documents)):
    collection.add(
        documents=documents[id],
        ids=[str(ids)]
    )
    id = id + 1
    ids = ids+1
    print(ids)
print("Data successfully added to ChromaDB!")

2104
2105
2106
2107
2108
2109
2110
2111
2112
2113
2114
2115
2116
2117
2118
2119
2120
2121
2122
2123
2124
2125
2126
2127
2128
2129
2130
2131
2132
2133
2134
2135
2136
2137
2138
2139
2140
2141
2142
2143
2144
2145
2146
2147
2148
2149
2150
2151
2152
2153
2154
2155
2156
2157
2158
2159
2160
2161
2162
2163
2164
2165
2166
2167
2168
2169
2170
2171
2172
2173
2174
2175
2176
2177
2178
2179
2180
2181
2182
2183
2184
2185
2186
2187
2188
2189
2190
2191
2192
2193
2194
2195
2196
2197
2198
2199
2200
2201
2202
2203
2204
2205
2206
2207
2208
2209
2210
2211
2212
2213
2214
2215
2216
2217
2218
2219
2220
2221
2222
2223
2224
2225
2226
2227
2228
2229
2230
2231
2232
2233
2234
2235
2236
2237
2238
2239
2240
2241
2242
2243
2244
2245
2246
2247
2248
2249
2250
2251
2252
2253
2254
2255
2256
2257
2258
2259
2260
2261
2262
2263
2264
2265
2266
2267
2268
2269
2270
2271
2272
2273
2274
2275
2276
2277
2278
2279
2280
2281
2282
2283
2284
2285
2286
2287
2288
2289
2290
2291
2292
2293
2294
2295
2296
2297
2298
2299
2300
2301
2302
2303


IndexError: list index out of range

In [123]:
print(ids)

2103
